In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.read_csv("/content/adult.csv")

X = df.drop("class", axis=1)
y = (df["class"] == ">50K").astype(int)

print(df.head())
print("Dataset shape: ", df.shape)

   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   34    Private  198693          10th              6       Never-married   

          occupation   relationship   race   sex  capital-gain  capital-loss  \
0  Machine-op-inspct      Own-child  Black  Male             0             0   
1    Farming-fishing        Husband  White  Male             0             0   
2    Protective-serv        Husband  White  Male             0             0   
3  Machine-op-inspct        Husband  Black  Male          7688             0   
4      Other-service  Not-in-family  White  Male             0             0   

   hours-per-week native-country  class  
0       

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (36140, 14)
X_test: (9035, 14)
y_train: (36140,)
y_test: (9035,)


In [ ]:
scaler = StandardScaler()
imp = SimpleImputer(strategy="median")

X_all = scaler.fit_transform(
    imp.fit_transform(X.select_dtypes(include=np.number))
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all,
    y,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

scores = cross_val_score(model, X_tr, y_tr, cv=5)

print("CV Scores: ", scores)
print("Mean Score: ", scores.mean())

CV Scores: [0.80743692 0.80932704 0.81213105 0.81213105 0.80386659]
Mean Score: 0.8089785290165586


In [9]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

model = LogisticRegression(max_iter=1000)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

scores = cross_validate(pipe, X_tr, y_tr, cv=5)

print("CV Scores:", scores["test_score"])
print("Mean Score:", scores["test_score"].mean())

CV Scores: [0.85210293 0.84463199 0.84532374 0.84615385 0.85044272]
Mean Score: 0.8477310459324847
